### Import Library

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load Dataset

In [3]:
df = pd.read_csv('US_Accidents_March23.csv')
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


### Inspeksi Awal Kondisi Data

In [4]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 46 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     object 
 1   Source                 object 
 2   Severity               int64  
 3   Start_Time             object 
 4   End_Time               object 
 5   Start_Lat              float64
 6   Start_Lng              float64
 7   End_Lat                float64
 8   End_Lng                float64
 9   Distance(mi)           float64
 10  Description            object 
 11  Street                 object 
 12  City                   object 
 13  County                 object 
 14  State                  object 
 15  Zipcode                object 
 16  Country                object 
 17  Timezone               object 
 18  Airport_Code           object 
 19  Weather_Timestamp      object 
 20  Temperature(F)         float64
 21  Wind_Chill(F)          float64
 22  Humidity(%)       

ID                             0
Source                         0
Severity                       0
Start_Time                     0
End_Time                       0
Start_Lat                      0
Start_Lng                      0
End_Lat                  3402762
End_Lng                  3402762
Distance(mi)                   0
Description                    5
Street                     10869
City                         253
County                         0
State                          0
Zipcode                     1915
Country                        0
Timezone                    7808
Airport_Code               22635
Weather_Timestamp         120228
Temperature(F)            163853
Wind_Chill(F)            1999019
Humidity(%)               174144
Pressure(in)              140679
Visibility(mi)            177098
Wind_Direction            175206
Wind_Speed(mph)           571233
Precipitation(in)        2203586
Weather_Condition         173459
Amenity                        0
Bump      

### Pembersihan Data (Data Cleaning & Missing Values)

In [5]:
# 1. Copy data biar dataframe aslinya tetap aman kalau ada salah eksekusi
df_clean = df.copy()

# 2. Drop kolom yang terlalu banyak missing values
cols_to_drop = ['End_Lat', 'End_Lng', 'Wind_Chill(F)']
df_clean = df_clean.drop(columns=cols_to_drop)

# 3. Imputasi kolom numerik (cuaca) dengan nilai Median
num_cols_to_impute = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)']
for col in num_cols_to_impute:
    # Memastikan kolom numerik diisi dengan nilai tengahnya
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 4. Hapus sisa baris yang masih memiliki missing values (untuk kolom minoritas)
df_clean = df_clean.dropna()

# 5. Cek lagi hasilnya untuk memastikan sudah bersih 100%
print(df_clean.isnull().sum())
print(f"\nSisa baris data yang siap dilatih: {len(df_clean)}")

ID                       0
Source                   0
Severity                 0
Start_Time               0
End_Time                 0
Start_Lat                0
Start_Lng                0
Distance(mi)             0
Description              0
Street                   0
City                     0
County                   0
State                    0
Zipcode                  0
Country                  0
Timezone                 0
Airport_Code             0
Weather_Timestamp        0
Temperature(F)           0
Humidity(%)              0
Pressure(in)             0
Visibility(mi)           0
Wind_Direction           0
Wind_Speed(mph)          0
Precipitation(in)        0
Weather_Condition        0
Amenity                  0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Roundabout               0
Station                  0
Stop                     0
Traffic_Calming          0
T

### Penanganan Outliers (Capping dengan IQR)

In [6]:
# Membuat fungsi untuk menghitung batas IQR dan melakukan Capping
def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Menerapkan batas bawah dan atas (Capping)
    dataframe[column] = np.where(dataframe[column] > upper_bound, upper_bound,
                        np.where(dataframe[column] < lower_bound, lower_bound, dataframe[column]))
    return dataframe

# Daftar kolom cuaca numerik yang akan ditangani outlier-nya
outlier_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)']

# Eksekusi capping untuk setiap kolom yang sudah ditentukan
for col in outlier_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

print("Handling Outliers dengan metode Capping selesai dieksekusi!")

Handling Outliers dengan metode Capping selesai dieksekusi!


### Transformasi Data (Encoding)

In [7]:
from sklearn.preprocessing import LabelEncoder

# 1. Konversi kolom boolean (True/False) menjadi nilai biner (1/0)
bool_cols = df_clean.select_dtypes(include='bool').columns
for col in bool_cols:
    df_clean[col] = df_clean[col].astype(int)

# 2. Label Encoding untuk kolom objek (teks/kategorikal)
label_cols = df_clean.select_dtypes(include='object').columns
le = LabelEncoder()

for col in label_cols:
    # Memastikan semua data bertipe string sebelum di-encode
    df_clean[col] = df_clean[col].astype(str)
    df_clean[col] = le.fit_transform(df_clean[col])

print("Proses Encoding selesai! Seluruh dataset kini berformat numerik.")
# Menampilkan 5 baris pertama untuk memastikan hasil transformasi
df_clean.head()

Proses Encoding selesai! Seluruh dataset kini berformat numerik.


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
5,3599383,1,3,2,1,40.100590,-82.925194,0.01,316742,277738,...,0,0,0,0,0,0,0,0,0,0
9,0,1,3,4,2,40.100590,-82.925194,0.01,2161273,277738,...,0,0,0,0,0,0,0,0,0,0
11,192495,1,3,5,3,39.932709,-82.830910,0.01,2084026,197598,...,0,0,0,0,0,0,0,0,0,0
14,479864,1,2,6,4,39.972038,-82.913521,0.01,460123,174757,...,0,0,0,0,1,0,0,0,0,0
20,1056849,1,2,7,5,40.052509,-82.882332,0.00,170239,156932,...,0,0,0,0,0,0,0,0,0,0


### Tahap Pemodelan

In [9]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import pickle

# 1. Pisahkan Fitur (X) dan Target (y)
# Target prediksi kita adalah kolom 'Severity'
X = df_clean.drop(columns=['Severity'])
y = df_clean['Severity']

# Catatan: Jika label target Severity dimulai dari 1-4, LightGBM multi-class memerlukan index mulai dari 0 (0-3).
# Kita kurangi 1 secara dinamis jika nilai minimumnya adalah 1.
if y.min() == 1:
    y = y - 1

# 2. Split data menjadi Train dan Test dengan proporsi 80:20
print("Sedang memisahkan data menjadi Train Set dan Test Set...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"-> Jumlah data training: {X_train.shape[0]} baris")
print(f"-> Jumlah data testing: {X_test.shape[0]} baris")

# 3. Inisialisasi dan Pelatihan Model LightGBM Classifier
print("\nMemulai pelatihan model LightGBM... (Proses ini memakan waktu beberapa menit, mohon tunggu)")
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1  # Menggunakan seluruh core CPU laptop agar proses komputasi maksimal & cepat
)

model.fit(X_train, y_train)
print("Pelatihan model LightGBM selesai dengan sukses!")

# 4. Ekspor model langsung menjadi file .pkl sesuai instruksi tugas
print("\nMengekspor model ke dalam format file .pkl...")
with open('model_us_accidents.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Selesai! File 'model_us_accidents.pkl' berhasil dibuat di direktori kamu.")

Sedang memisahkan data menjadi Train Set dan Test Set...
-> Jumlah data training: 4321009 baris
-> Jumlah data testing: 1080253 baris

Memulai pelatihan model LightGBM... (Proses ini memakan waktu beberapa menit, mohon tunggu)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.234832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4163
[LightGBM] [Info] Number of data points in the train set: 4321009, number of used features: 40
[LightGBM] [Info] Start training from score -4.459681
[LightGBM] [Info] Start training from score -0.167669
[LightGBM] [Info] Start training from score -2.130752
[LightGBM] [Info] Start training from score -3.727507
Pelatihan model LightGBM selesai dengan sukses!

Mengekspor model ke dalam format file .pkl...
Selesai! File 'model_us_accidents.pkl' berhasil dibuat di direktori kamu.


### Evaluasi Model

In [10]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Gunakan model yang sudah dilatih untuk menebak data testing
print("Memproses prediksi pada data testing...")
y_pred = model.predict(X_test)

# 2. Hitung dan tampilkan hasil evaluasinya
akurasi = accuracy_score(y_test, y_pred)
print("\n=== HASIL EVALUASI MODEL LIGHTGBM ===")
print(f"Akurasi Keseluruhan: {akurasi * 100:.2f}%\n")
    
print("Detail Classification Report:")
print(classification_report(y_test, y_pred))

Memproses prediksi pada data testing...

=== HASIL EVALUASI MODEL LIGHTGBM ===
Akurasi Keseluruhan: 93.65%

Detail Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.66      0.72     12494
           1       0.95      0.98      0.96    913498
           2       0.84      0.78      0.81    128278
           3       0.88      0.34      0.49     25983

    accuracy                           0.94   1080253
   macro avg       0.87      0.69      0.75   1080253
weighted avg       0.93      0.94      0.93   1080253

